# **ML Pipeline: FashionMNIST Classification (PyTorch)**

## Pipeline Map and Stage Tracking

The objective is not only to train a model, but also to demonstrate clear engineering thinking, modular design, and reproducible workflow execution.

1. Stage 0: Setup and reusable utilities
2. Stage 1: Data loading and preprocessing
3. Stage 2: Data understanding (features, labels, batches)
4. Stage 3: Model, loss, and optimizer configuration
5. Stage 4: Training loop
6. Stage 5: Evaluation on test data

Interview tip: explain each stage as a decision point in the ML lifecycle (data -> model -> optimization -> validation).

## **Stage 0 - Import Libraries and Define Reusable Building Blocks**

The next code cell contains the reusable core of the pipeline.

**What this stage achieves:**

- imports all required libraries once
- defines constants for shared configuration (batch size, learning rate, number of classes)
- creates reusable helper functions for each major operation

**Why this is professional:**

- prevents copy/paste logic
- makes experimentation easier (change one function, keep the rest unchanged)
- keeps the notebook consistent with production-oriented coding style

**Main utility functions and Reusable helper groups in this notebook:**

- **data utilities**: `create_transform`, `load_fashion_mnist` — load and process data
- **data loading**: `create_loader` — create mini-batches
- **model/loss/optimizer**: `build_model`, `create_loss_fn`, `create_optimizer` — build the model
- **training**: `train_one_epoch` — run one training epoch
- **evaluation**: `evaluate_accuracy` — check accuracy
- **debugging**: `inspect_sample` — look at raw data samples

## **1. Imports**

In [ ]:
import torch  # Core PyTorch package for tensor operations and training.
import torch.nn as nn  # Neural-network layers and loss functions.
from torch.utils.data import DataLoader  # Efficient mini-batch data loading utility.
from torchvision import datasets, transforms  # Vision datasets and preprocessing transforms.

## **2. hyperparameters / config**

In [ ]:
DEFAULT_DATA_ROOT = "data"  # Default local folder for dataset storage.
DEFAULT_BATCH_SIZE = 64  # Standard mini-batch size used across the notebook.
DEFAULT_LR = 1e-3  # Default learning rate for optimizer creation.
NUM_CLASSES = 10  # Number of FashionMNIST classes.

## **3. Functions**

### `0-3.1 - create_transform()` - Image Preprocessing Pipeline

In [ ]:
def create_transform():  # Build the default preprocessing pipeline.
    return transforms.ToTensor()  # Convert images to tensors and scale pixels to [0, 1].

**Purpose:** Create a reusable preprocessing transform for converting raw images into model-ready tensors.

**What it does:**
- Converts PIL images or NumPy arrays → PyTorch tensors
- Normalizes pixel values from [0, 255] → [0.0, 1.0]
- Returns a `transforms.ToTensor()` object

**Why this function:**
- Single source of truth for preprocessing logic
- Easy to extend later (add augmentation, normalization, etc.)
- Ensures consistency between training and evaluation data

**Key Parameters:** None (uses default `transforms.ToTensor()`)

**Returns:** 
- `transforms.ToTensor()` - A transform function ready to be passed to dataset constructors

---
### `0.3.2 - load_fashion_mnist()` - Dataset Loader


In [ ]:
def load_fashion_mnist(data_root=DEFAULT_DATA_ROOT, transform=None):  # Load train/test splits with a shared transform.
    if transform is None:  # If caller did not provide a transform, create one.
        transform = create_transform()  # Use the default tensor conversion transform.

    train_ds = datasets.FashionMNIST(  # Initialize the training dataset object.
        root=data_root,  # Set dataset root directory.
        train=True,  # Select the training split.
        download=True,  # Download dataset if not present locally.
        transform=transform,  # Apply preprocessing when each sample is accessed.
    )  # Finish training dataset construction.
    test_ds = datasets.FashionMNIST(  # Initialize the test dataset object.
        root=data_root,  # Set dataset root directory.
        train=False,  # Select the test split.
        download=True,  # Download dataset if not present locally.
        transform=transform,  # Apply the same preprocessing for fair evaluation.
    )  # Finish test dataset construction.
    return train_ds, test_ds  # Return both dataset splits to the caller.

**Purpose:** Load FashionMNIST training and test datasets with consistent preprocessing.

**What it does:**
- Downloads FashionMNIST dataset (if not already present locally)
- Loads training split (60,000 images)
- Loads test split (10,000 images)
- Applies the same transform to both splits for fair evaluation

**Why this function:**
- Encapsulates dataset loading logic
- Ensures preprocessing parity between train/test
- Easy to call from anywhere in the pipeline
- Auto-downloads dataset on first run

**Key Parameters:**
- `data_root` - Directory where dataset will be stored (default: "data")
- `transform` - Preprocessing transform to apply (default: creates new one via `create_transform()`)

**Returns:**
- `train_ds` - Training dataset object with 60,000 samples
- `test_ds` - Test dataset object with 10,000 samples

---
### `0.3.3 - create_loader()` - Mini-Batch Iterator

In [ ]:
def create_loader(dataset, batch_size=DEFAULT_BATCH_SIZE, shuffle=False):  # Build a reusable DataLoader for any dataset.
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)  # Return batched iterable dataset.

**Purpose:** Wrap a dataset in a DataLoader for efficient mini-batch iteration.

**What it does:**
- Creates mini-batches of configurable size (default: 64)
- Optionally shuffles samples (useful for training)
- Returns an iterable DataLoader object

**Why this function:**
- Encapsulates DataLoader creation logic
- Enables vectorized operations (GPU efficiency)
- Consistent batch handling throughout pipeline
- Reusable for any dataset

**Key Parameters:**
- `dataset` - PyTorch dataset object (e.g., FashionMNIST)
- `batch_size` - Number of samples per batch (default: 64)
- `shuffle` - Randomize batch order (default: False)
  - Use `True` for training (prevents overfitting)
  - Use `False` for testing (reproducible evaluation)

**Returns:**
- `DataLoader` - An iterable that yields batches of (images, labels)

----
### `0.3.4 -build_model()` - Neural Network Architecture

In [ ]:
def build_model(input_shape=(1, 28, 28), hidden_size=128, num_classes=NUM_CLASSES):  # Create a simple feed-forward classifier.
    _, h, w = input_shape  # Extract image height and width from input shape.
    return nn.Sequential(  # Stack layers in sequence for the forward pass.
        nn.Flatten(),  # Flatten [C, H, W] image into a 1D feature vector.
        nn.Linear(h * w, hidden_size),  # Project flattened input to hidden representation.
        nn.ReLU(),  # Add non-linearity to improve expressiveness.
        nn.Linear(hidden_size, num_classes),  # Map hidden features to class logits.
    )  # Return the assembled model.

**Purpose:** Create a simple feedforward neural network for image classification.

**What it does:**
- Flattens 2D images (1×28×28) into 1D vectors (784-dim)
- Adds dense hidden layer (784 → hidden_size, default: 128)
- Adds ReLU activation (non-linearity)
- Adds output layer (hidden_size → 10 classes, logits)

**Why this function:**
- Encapsulates architecture definition
- Easy to modify capacity (adjust `hidden_size`)
- Can be replaced with CNN or deeper models later
- Clean separation of architecture from training

**Architecture Visualization:**
```
Input [1×28×28] 
  ↓ Flatten
→ [784] 
  ↓ Linear(784→128)
→ [128] 
  ↓ ReLU
→ [128] (with non-linearity)
  ↓ Linear(128→10)
→ [10] (logits for 10 classes)
```

**Key Parameters:**
- `input_shape` - Image dimensions (default: (1, 28, 28) for FashionMNIST)
- `hidden_size` - Number of hidden neurons (default: 128)
- `num_classes` - Number of output classes (default: 10)

**Returns:**
- `nn.Sequential` - Compiled model ready for training

---
### `0-3.5 - create_loss_fn()` - Classification Loss Function


In [ ]:
def create_loss_fn():  # Create the classification loss function.
    return nn.CrossEntropyLoss()  # Standard loss for multi-class classification with logits.

**Purpose:** Create the optimization objective that measures prediction error.

**What it does:**
- Returns `CrossEntropyLoss` - the standard loss for multi-class classification
- Internally applies softmax (converts logits → probabilities)
- Applies negative log likelihood
- Produces a scalar loss value

**Why this function:**
- `CrossEntropyLoss` is the industry standard for classification
- Expects raw logits (no softmax in model output)
- Compatible with integer class labels (not one-hot encoded)
- Encapsulates loss creation for easy modification

**Mathematical Insight:**
```
CrossEntropyLoss = -log(softmax(logits)[true_class])
```
- High loss when model assigns low probability to correct class
- Low loss when model assigns high probability to correct class

**Key Parameters:** None (uses defaults)

**Returns:**
- `nn.CrossEntropyLoss()` - Loss function object ready for training loop

### `0-3.6 - create_optimizer()` - Gradient Descent Optimizer


In [ ]:
def create_optimizer(model, lr=DEFAULT_LR):  # Create optimizer for model parameters.
    return torch.optim.Adam(model.parameters(), lr=lr)  # Adam optimizer with configurable learning rate.

**Purpose:** Create the optimizer algorithm that updates model weights during training.

**What it does:**
- Creates Adam optimizer with configurable learning rate
- Adam = "Adaptive Moment Estimation" (modern, adaptive gradient descent)
- Maintains adaptive learning rates per parameter
- Includes momentum for smooth convergence

**Why Adam:**
- **Adaptive** - Different learning rates for different parameters
- **Momentum** - Helps escape local minima and smooth updates
- **Robust** - Works well with default hyperparameters
- **Industry Standard** - Used in most modern deep learning

**Why this function:**
- Encapsulates optimizer creation
- Easy to switch to SGD or other optimizers later
- Single place to configure learning rate

**Key Parameters:**
- `model` - Neural network whose parameters will be optimized
- `lr` - Learning rate (default: 0.001)
  - Controls update step magnitude
  - Too high → divergence, too low → slow convergence

**Returns:**
- `torch.optim.Adam` - Optimizer object ready for training loop

**Interview Note:** Mention that learning rate is the first hyperparameter to tune if training doesn't work.

------
### `0-3.7 - train_one_epoch()` - Single Training Epoch

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device="cpu"):  # Train model for one full pass over the loader.
    model.train()  # Enable training mode (affects layers like dropout/batchnorm).
    running_loss = 0.0  # Accumulate sample-weighted loss over the epoch.

    for images, labels in loader:  # Iterate through all mini-batches.
        images, labels = images.to(device), labels.to(device)  # Move batch tensors to selected device.
        outputs = model(images)  # Compute forward-pass logits.
        loss = loss_fn(outputs, labels)  # Compute loss against true labels.

        optimizer.zero_grad()  # Reset gradients from previous iteration.
        loss.backward()  # Compute gradients for the current iteration.
        optimizer.step()  # Update model parameters using gradients.

        running_loss += loss.item() * labels.size(0)  # Accumulate total loss for this batch.

    return running_loss / len(loader.dataset)  # Return average loss per sample for the epoch.

**Purpose:** Run one complete pass through the training data, updating model weights.

**What it does:**
1. Enables training mode (`model.train()`)
2. Iterates through mini-batches from the DataLoader
3. Forward pass: computes predictions (`logits = model(images)`)
4. Backward pass: computes gradients (`loss.backward()`)
5. Updates weights using optimizer (`optimizer.step()`)
6. Accumulates and returns average loss

**Training Loop Sequence (for each batch):**
```
1. Forward:     logits = model(images)
2. Loss:        loss = criterion(logits, labels)
3. Backward:    loss.backward()  # Compute gradients
4. Update:      optimizer.step()  # Apply gradients
5. Accumulate:  running_loss += loss
```

**Key Parameters:**
- `model` - Neural network being trained
- `loader` - DataLoader yielding mini-batches
- `loss_fn` - Loss function (e.g., CrossEntropyLoss)
- `optimizer` - Optimizer (e.g., Adam)
- `device` - Where to move data (CPU or CUDA)

**Returns:**
- `avg_loss` - Average loss per sample for the entire epoch
  - Lower loss = better training progress
  - Monitor this to detect divergence or convergence

**Why Encapsulate:**
- Reusable for multiple epochs
- Clean separation of training logic
- Easy to modify training behavior centrally

-----
### `0-3.8 - evaluate_accuracy()` - Model Evaluation


In [ ]:
def evaluate_accuracy(model, loader, device="cpu"):  # Evaluate classification accuracy on a loader.
    model.eval()  # Switch model to evaluation mode.
    correct = 0  # Count correctly predicted samples.
    total = 0  # Count total evaluated samples.

    with torch.no_grad():  # Disable gradient tracking for inference efficiency.
        for images, labels in loader:  # Iterate through evaluation mini-batches.
            images, labels = images.to(device), labels.to(device)  # Move batch tensors to selected device.
            outputs = model(images)  # Compute logits for current batch.
            predicted = outputs.argmax(dim=1)  # Convert logits to predicted class indices.
            correct += (predicted == labels).sum().item()  # Add number of correct predictions.
            total += labels.size(0)  # Add number of samples in batch.

    return correct / total if total else 0.0  # Return accuracy, guarding against empty loader.

**Purpose:** Measure how well the trained model generalizes to unseen data.

**What it does:**
1. Switches model to evaluation mode (`model.eval()`)
2. Disables gradient computation (`torch.no_grad()`) for efficiency
3. Iterates through all test mini-batches
4. Counts correct predictions vs. total samples
5. Returns accuracy = correct / total

**Evaluation Mode Details:**
- `model.eval()` disables training-specific behaviors (dropout, batch norm)
- `torch.no_grad()` prevents gradient tracking → saves memory & speed
- No parameter updates during evaluation (read-only pass)

**Key Metrics:**
- `predicted = outputs.argmax(dim=1)` - Take class with highest logit as prediction
- `correct` - Count samples where prediction matches ground truth
- `accuracy = correct / total` - Fraction of correct predictions

**Key Parameters:**
- `model` - Trained neural network in evaluation mode
- `loader` - DataLoader for test or validation data
- `device` - CPU or CUDA

**Returns:**
- `accuracy` (0.0 to 1.0) - Fraction of correct predictions on the dataset
  - 0.5 = random guessing (for 10 classes, baseline is 0.1)
  - 0.95 = excellent performance
  - Used to compare models and detect overfitting

**Interview Talking Points:**
- This is the "real" metric - training accuracy can lie (overfitting)
- Test accuracy tells if model generalizes to unseen data
- Gap between train/test accuracy indicates overfitting

--------
### `0.3.9 - inspect_sample()` - Sample Debugging & Inspection


In [ ]:
def inspect_sample(dataset, index=0):  # Inspect one sample for debugging and explanation.
    image, label = dataset[index]  # Retrieve image tensor and label at selected index.
    return {  # Return a structured dictionary with sample metadata.
        "image": image,  # Raw image tensor.
        "label": label,  # Numeric class id.
        "shape": tuple(image.shape),  # Tensor shape for quick validation.
        "class_name": dataset.classes[label],  # Human-readable class name.
        "class_to_idx": dataset.class_to_idx,  # Full mapping from class names to indices.
    }  # End sample-info dictionary.

**Purpose:** Extract and return detailed information about a single sample for visual inspection and debugging.

**What it does:**
1. Retrieves one sample from the dataset at specified index
2. Unpacks image tensor and label
3. Returns structured dictionary with:
   - Raw image tensor
   - Numeric label (class index)
   - Tensor shape (for validation)
   - Human-readable class name
   - Full class-to-index mapping dictionary

**Why this function:**
- Debugging tool to verify data is loaded correctly
- Spot-check samples before training
- Confirm label-to-class mapping is correct
- Detect data loading issues early

**Sample Output:**
```python
{
    'image': tensor([...]),           # Preprocessed image as tensor
    'label': 2,                       # Numeric class ID
    'shape': (1, 28, 28),           # Image dimensions
    'class_name': 'Pullover',       # Human-readable label
    'class_to_idx': {...}           # Full mapping dict
}
```

**Key Parameters:**
- `dataset` - PyTorch dataset object (e.g., FashionMNIST)
- `index` - Which sample to inspect (default: 0 = first sample)

**Returns:**
- Dictionary containing:
  - `'image'` - Tensor representation
  - `'label'` - Integer class ID (0-9 for FashionMNIST)
  - `'shape'` - Tuple of dimensions
  - `'class_name'` - String label (e.g., "T-shirt/top")
  - `'class_to_idx'` - Full class mapping

**Interview Talking Point:**
"Before training, always inspect a few samples to ensure your data pipeline is correct. This catches bugs early."

---


## Stage 1 - Preprocessing: Convert Images to Tensors

`transforms.ToTensor()` is the bridge between raw image objects and model-ready numeric tensors.

Detailed effect:

- input format: PIL image or NumPy array
- output format: `torch.Tensor`
- value scaling: `[0, 255]` -> `[0.0, 1.0]`

In interview discussion, this is where you show that you understand data representation, not just model APIs.

In [ ]:
transform = create_transform()  # Create reusable image-to-tensor preprocessing transform.
# Raw data (image) -> tensor in [0, 1]  # Explain the effect of the transform.

### Why Tensor Conversion Matters

Neural networks consume tensors, so preprocessing must convert each sample into numeric form before training.

After applying `ToTensor()`:

- data type becomes `torch.Tensor`
- pixel range becomes normalized to `[0.0, 1.0]`
- FashionMNIST sample shape is typically `[1, 28, 28]`

This normalization stabilizes optimization and gives consistent numeric scale across samples.

### Reusable Preprocessing Pattern

Preprocessing is passed into dataset constructors rather than applied manually in random cells.

Benefits of this design:

- single source of truth for transformations
- no mismatch between train/test preprocessing
- easier migration to augmentation pipelines later

In short: data preparation remains explicit, controlled, and reproducible.

---


## **Stage 1 - Load Datasets**

### Stage 1.1 - Load Training Dataset

The next code cell loads the training split using the shared transform.

What to mention in interview:

- this split is used to update model parameters
- transformation is attached at dataset level
- loading logic is reusable and parameterized

In [ ]:
train_dataset, _ = load_fashion_mnist(data_root=DEFAULT_DATA_ROOT, transform=transform)  # Load only training split using shared transform.

### Stage 1.2 - Load Test Dataset

The test split is loaded with the same transform to ensure fair and consistent evaluation.

Design principle: preprocessing parity between training and testing prevents evaluation bias caused by mismatched input pipelines.

In [ ]:
_, test_dataset = load_fashion_mnist(data_root=DEFAULT_DATA_ROOT, transform=transform)  # Load only test split using the same transform.

### Stage 1 Checkpoint

Both datasets are now available and aligned on the same preprocessing settings.

This completes the data-loading phase and prepares the notebook for semantic data validation.

## **Stage 2 - Understand Features and Labels**

Before model training, we verify the semantic meaning of data tensors and targets.

A FashionMNIST sample is a tuple: `(image_tensor, class_index)`.

- Feature: image tensor (model input)
- Label: integer class id in the range 0-9

Why this stage is essential:

- catches label/shape mistakes early
- validates dataset class mapping
- strengthens confidence in downstream training metrics

In [ ]:
transform = create_transform()  # Recreate transform for explicit stage-level clarity.

### Class Mapping

`train_dataset.classes` gives label-to-name mapping as an ordered list.
`train_dataset.class_to_idx` gives class-to-label mapping as a dictionary.

These two views should be checked together to avoid silent label-interpretation errors.

In [ ]:
class_names = train_dataset.classes  # Read ordered class names from dataset metadata.
print(class_names[0])  # Display first class label (T-shirt/top).

T-shirt/top


In [ ]:
print(train_dataset.class_to_idx)  # Show mapping from class names to numeric labels.

{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}


### Sample Inspection

The next code cell inspects one training sample and prints:

- tensor shape
- numeric label
- human-readable class name

This is a practical verification step that should be performed before any training loop starts.

In [ ]:
sample = inspect_sample(train_dataset, index=0)  # Inspect the first training sample.
print("image.shape=", sample["shape"])  # Print tensor shape for validation.
print("label=", sample["label"])  # Print numeric class index.
print("class=", sample["class_name"])  # Print human-readable class name.

image.shape= torch.Size([1, 28, 28]) class=
label= 9
class= Ankle boot


### Stage 2 Summary

At this point, the supervised learning structure is validated end to end:

- Feature = image tensor
- Label = integer class index
- Class identity = derived from dataset mapping

Interview framing: this stage demonstrates data correctness discipline before optimization begins.

---

### Stage 2.1 - Mini-batches with DataLoader

DataLoader converts a dataset into iterable mini-batches.

Why mini-batches are used:

- efficient GPU/CPU utilization via vectorized operations
- smoother and more stable gradient updates
- memory control through configurable `batch_size`

This stage prepares data for scalable training, not just single-sample experimentation.

In [ ]:
sample = inspect_sample(train_dataset, index=0)  # Fetch one sample for detailed inspection.

print("Image shape:", sample["shape"])  # Show tensor dimensions.
print("Raw label (number):", sample["label"])  # Show numeric target id.
print("Image tensor type:", type(sample["image"]))  # Confirm sample type is torch.Tensor.

class_names = train_dataset.classes  # Read class-name list from dataset.
print("Class for this label:", sample["class_name"])  # Show resolved class name for the sample label.

print("\nLabel -> Class mapping:")  # Print heading for index-to-name mapping.
for idx, name in enumerate(class_names):  # Iterate over every class index and class name.
    print(f"{idx}: {name}")  # Print each index-name pair.

print("\nClass -> Label mapping (same info, reverse view):")  # Print heading for reverse mapping.
print(sample["class_to_idx"])  # Print dictionary mapping class names to indices.

Image shape: torch.Size([1, 28, 28])
Raw label (number): 9
Image tensor type: <class 'torch.Tensor'>
Class for this label: Ankle boot

Label -> Class mapping:
0: T-shirt/top
1: Trouser
2: Pullover
3: Dress
4: Coat
5: Sandal
6: Shirt
7: Sneaker
8: Bag
9: Ankle boot

Class -> Label mapping (same info, reverse view):
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}


In [ ]:
train_loader = create_loader(train_dataset, batch_size=DEFAULT_BATCH_SIZE, shuffle=True)  # Build shuffled training DataLoader.

for images, labels in train_loader:  # Iterate over training batches.
    print("Batch of images shape:", images.shape)  # Print image batch tensor shape.
    print("Batch of labels shape:", labels.shape)  # Print label batch tensor shape.
    break  # Stop after first batch for quick shape check.

Batch of images shape: torch.Size([64, 1, 28, 28])
Batch of labels shape: torch.Size([64])


### Batch Interpretation

For `images.shape = [64, 1, 28, 28]`:

- `64`: number of samples in the current mini-batch
- `1`: grayscale channel
- `28 x 28`: spatial dimensions

For `labels.shape = [64]`:

- one class index per image
- positional correspondence between `images[i]` and `labels[i]`

This confirms the data interface expected by the model and loss function.

In [ ]:
sample = inspect_sample(test_dataset, index=0)  # Inspect the first test sample.
print("Image shape:", sample["shape"])  # Print test-image shape.
print("Label:", sample["label"])  # Print test-image label index.
print("Image tensor type:", type(sample["image"]))  # Print object type for test image tensor.
print(sample["image"])  # Print raw tensor values for visual inspection.

Image shape: torch.Size([1, 28, 28])
Label: 9
Image tensor:
<class 'torch.Tensor'>
tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000

### Target Semantics

The label is the supervised target class index that the model must predict correctly.

### Note on One-Hot Encoding

In this pipeline, labels should remain integer class indices.

Reason: `CrossEntropyLoss` expects logits as predictions and integer class ids as targets.

One-hot encoding is only required when a specific custom loss or model design explicitly demands it.

### Data Stage Final Checklist

Before moving to model configuration, verify all of the following:

- features are tensors with expected shape
- labels are integer class indices
- label-to-class mapping is understood
- mini-batch loading is functional

Passing this checklist reduces debugging during optimization stages.

---

### Loss Function Demonstration

The next code example demonstrates the correct target format for `CrossEntropyLoss` and clarifies why one-hot labels are not required in this setup.

In [ ]:
criterion = create_loss_fn()  # Instantiate cross-entropy loss function.

# Correct target format: class index, not one-hot  # Clarify expected target format.
labels = torch.tensor([3])  # Create a sample target tensor with batch size 1.
outputs = torch.randn(1, NUM_CLASSES)  # Create fake logits for demonstration.

loss = criterion(outputs, labels)  # Compute loss between logits and class index target.

print("Outputs:", outputs)  # Print example logits.
print("Label:", labels)  # Print ground-truth label tensor.
print("Loss:", loss.item())  # Print scalar loss value.

### Stage 2 Close-Out

Data semantics, class mapping, and batch behavior are validated.

The pipeline is now ready for model definition and optimization.

---


## **Stage 3 - Model and Optimization Setup**

### Stage 3.1 - Create Training DataLoader

`train_loader` controls batching and randomization during training.

Using `shuffle=True` helps avoid fixed-order learning artifacts and improves robustness of stochastic optimization.

In [ ]:
train_loader = create_loader(train_dataset, batch_size=DEFAULT_BATCH_SIZE, shuffle=True)  # Rebuild training DataLoader for model training stage.

### Stage 3.2 - Build Model

We initialize a reusable baseline network for FashionMNIST classification.

This stage isolates architecture creation from training logic, which keeps experimentation clean and maintainable.

### Architecture Notes

Input: tensor with shape `[1, 28, 28]`
Flatten: converts spatial image to a 784-dimensional vector
Hidden layer: learns compact intermediate features
Output layer: produces 10 logits for class decision

This architecture is intentionally simple to keep focus on pipeline quality and training mechanics.

This baseline is ideal for interviews because it demonstrates complete ML flow without architectural noise.

The same reusable pipeline can later host CNN or deeper models by only changing `build_model()`.

In [ ]:
model = build_model()  # Instantiate baseline feed-forward classification model.

### Stage 3.3 - Define Loss Function

In [ ]:
loss_fn = create_loss_fn()  # Create training loss function used in optimization loop.

`CrossEntropyLoss` is the standard objective for multi-class classification with logits.

It compares predicted class distributions against integer targets and provides differentiable error for backpropagation.

### Stage 3.4 - Define Optimizer

In [ ]:
optimizer = create_optimizer(model, lr=DEFAULT_LR)  # Create Adam optimizer for model parameters.

`Adam` performs adaptive gradient-based parameter updates.

It is commonly used for fast and stable convergence in practical deep learning workflows.

### Hyperparameter Notes

- `lr` (learning rate) controls update magnitude
- very high `lr` can cause unstable or divergent training
- very low `lr` can make convergence too slow

In interviews, mention that learning rate is often the first hyperparameter to tune.

Stage 3 complete: data loader, model, loss function, and optimizer are fully initialized for training.

## **Stage 4 - Train the Model**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"  # Select GPU when available, otherwise CPU.
model = model.to(device)  # Move model weights to selected device.

train_loader = create_loader(train_dataset, batch_size=DEFAULT_BATCH_SIZE, shuffle=True)  # Ensure train loader is ready for this stage.
train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device=device)  # Run one full training epoch.
print(f"Train loss: {train_loss:.4f}")  # Print average training loss for the epoch.

The training cell runs one reusable epoch routine with the following sequence:

1. switch model to training mode
2. iterate over mini-batches
3. run forward pass
4. compute loss
5. compute gradients via backpropagation
6. update parameters via optimizer

The printed loss is the epoch-level average and serves as a primary optimization signal.

---


##  **Stage 5 - Evaluate on Test Data**

We build a deterministic test loader (`shuffle=False`) and evaluate generalization performance on unseen samples.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"  # Select GPU when available, otherwise CPU.
model = model.to(device)  # Ensure model is on the same device as evaluation tensors.

test_loader = create_loader(test_dataset, batch_size=DEFAULT_BATCH_SIZE, shuffle=False)  # Build deterministic test DataLoader.
accuracy = evaluate_accuracy(model, test_loader, device=device)  # Compute classification accuracy on test set.
print("Accuracy:", accuracy)  # Print final evaluation accuracy.

### Evaluation Notes

- `model.eval()` activates inference behavior
- `torch.no_grad()` disables gradient tracking for speed and memory efficiency
- accuracy is computed as `correct_predictions / total_samples`

This final metric is the outcome of the full pipeline and should be interpreted together with training loss trends.